In [1]:
import os
import pandas as pd
import numpy as np
import optuna
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler, OneHotEncoder, MinMaxScaler
from optuna.samplers import RandomSampler
from torchvision import transforms
#from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
#from torch.utils.data import DataLoader, random_split
from torch.nn import CrossEntropyLoss
from torch.optim import Adam

from utils import read_images, get_scaler_engagement, get_score_engagement, categorize_engagement_score, get_normalize_images
from utilsOptuna import optuna_init
from utilsDataset import img_Dataset
from utilsNN import FCNN
from utilsTrain import train_model

In [2]:
# load dataset
poi_data = pd.read_csv("poi_dataset.csv")
# TODO function to apply modifications to columns numTags, categories, Likes_Dislikes (easier to deal with ColumnTransformer)

In [3]:
def process_data(df):
    df['NumTags'] = df['tags'].apply(eval).apply(len)
    df['categories'] = df['categories'].apply(eval)
    df['Likes_Dislikes'] = df['Likes'] - df['Dislikes']
    return df
poi_data_processed = process_data(poi_data)

In [4]:
# split data into train, val and test datasets
df_train, df_test = train_test_split(poi_data_processed, test_size = 0.2, random_state = 42)
df_train, df_val = train_test_split(df_train, test_size = 0.2, random_state = 42)
print(f'Number of samples.')
print(f'Train dataset: {df_train.shape[0]}')
print(f'Validation dataset: {df_val.shape[0]}')
print(f'Test dataset: {df_test.shape[0]}')

Number of samples.
Train dataset: 1004
Validation dataset: 251
Test dataset: 314


In [37]:
class MultiLabelBinarizerWrapper(BaseEstimator, TransformerMixin):
    """
    Wrapper for MultiLabelBinarizer to work with ColumnTransformer
    """
    def __init__(self):
        self.mlb = MultiLabelBinarizer()
    
    def fit(self, X, y=None):
        X_series = self._safe_squeeze(X)
        self.mlb.fit(X_series)
        return self
    
    def transform(self, X):
        X_series = self._safe_squeeze(X)
        return self.mlb.transform(X_series)
    
    def fit_transform(self, X, y=None):
        return self.fit(X).transform(X)
    
    def get_feature_names_out(self, input_features=None):
        return self.mlb.classes_

    def _safe_squeeze(self, X):
        """
        Safely convert one column dataframe to pd.Series with squeeze function
        Only squeezes if X is a one-column DataFrame, otherwise raises error.
        """
        # Check if X is a DataFrame
        if not hasattr(X, 'iloc') or not hasattr(X, 'shape'):
            raise ValueError(
                f"Input must be a DataFrame. Got {type(X)} instead. "
                "Make sure this transformer is used in ColumnTransformer with proper column selection."
            )
        
        # Check if DataFrame has exactly one column
        if X.shape[1] != 1:
            raise ValueError(
                f"Expected 1 column, but got {X.shape[1]} columns. "
                "This transformer should only be applied to a single column. "
                "Check your ColumnTransformer configuration."
            )
        
        # Extract the single column
        return X.squeeze()

In [98]:
import statistics
class TargetFeature(BaseEstimator, TransformerMixin):
    def __init__(self, col1_name, col2_name):
        self.col1 = col1_name
        self.col2 = col2_name
        self.scaler1 = MinMaxScaler()
        self.scaler2 = MinMaxScaler()
        self.threshold = None
    
    def fit(self, X, y=None):
        #self.scaler1.fit(X[[self.col1]])
        #self.scaler2.fit(X[[self.col2]])
        col1_scaled = self.scaler1.fit_transform(X[[self.col1]])
        col2_scaled = self.scaler2.fit_transform(X[[self.col2]])
        target = (col1_scaled + col2_scaled) / 2.0
        self.threshold = statistics.median(target)
        return self
    
    def transform(self, X):
        col1_scaled = self.scaler1.transform(X[[self.col1]])
        col2_scaled = self.scaler2.transform(X[[self.col2]])
        target = (col1_scaled + col2_scaled) / 2.0
        return np.array([1 if x[0] > self.threshold else 0 for x in target])

In [104]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

preproc_target = TargetFeature(col1_name='Visits', col2_name='Likes_Dislikes')
preproc_explanatory = ColumnTransformer(
    transformers=[
        ('numerical', StandardScaler(), ['xps', 'locationLon', 'locationLat', 'NumTags']),
        ('categories', MultiLabelBinarizerWrapper(), ['categories']), 
        ('tier',OneHotEncoder(sparse_output=False),['tier'])
    ],
    #remainder="passthrough"
    remainder="drop"
)
preproc_union = FeatureUnion([
    ('explanatory', preproc_explanatory),
    ('target', preproc_target)
])

# Fit and transform
X_train = preproc_explanatory.fit_transform(df_train)
y_train = preproc_target.fit_transform(df_train)
X_val = preproc_explanatory.transform(df_val)
y_val = preproc_target.transform(df_val)

model = RandomForestClassifier()

model.fit(X_train, y_train)
model.predict(X_val)
#df_train_processed = pd.DataFrame(preproc_union.fit_transform(df_train), columns = 
#df_val_processed = preproc_union.transform(df_val)
#df_test_processed = preproc_union.transform(df_test)

ModuleNotFoundError: No module named 'xgboost'

In [15]:
# Preprocessing: get fit from train data
# get MinMaxScaler for engagement rows (This will be used for engagement calculation in different dataset
scaler_engagement = get_scaler_engagement(df_train)
# Normalize xps, locationLon, locationLat, numTags
scaler_features = StandardScaler().fit(df_train[['xps','locationLon','locationLat']])
# one hot encoder for categories
# TODO what happens if categoriy in val not present in train
onehot_encoder_categories = MultiLabelBinarizer().fit(df_train['categories'].apply(eval))
# One hot encoder for tier
onehot_encoder_tier = OneHotEncoder(sparse_output=False).fit(df_train[['tier']])
onehot_encoder_tier = OneHotEncoder(sparse_output=False).fit(df_train[['tier']])

In [7]:
def processdata(df):
    # This could be copy into Dataset. And make different Datasets with different names????
    """
    Data processing steps before being used in the model.
    Same steps are applied to train, validation and test datasets.
    Processing include:
    - Calculate engagement feature (low, medium, high)
    """
    df['score'] = get_score_engagement(df, scaler_engagement)
    df['engagement'] = df['score'].apply(categorize_engagement_score)
    # Scale features xps, locationLon, locationLat
    df[['xps','locationLon','locationLat']] = scaler_features.transform(df[['xps','locationLon','locationLat']])
    # One hot encoder for categories
    #categories_one_hot = onehot_encoder_categories.transform(df['categories'].apply(eval))
    #df_categories_one_hot = pd.DataFrame(categories_one_hot, columns=onehot_encoder_categories.classes_)
    #df = pd.concat([df, df_categories_one_hot], axis=1)
    return df

In [ ]:
def processdata(df):
  """
  Data processing steps before being used in the model.
  Same steps are applied to train, validation and test datasets.
  Processing include:
  - Calculate number of tags
  - One hot encoding for categories
  - Calculate engagement feature (low, medium, high)
  - Scale quantitative features
  - One hot encoding for tier
  - Remove features
  """
  df.index = range(df.shape[0])
  # Feature for number of tags
  df['NumTags'] = df['tags'].apply(eval).apply(len)
  # One hot encoder for categories
  categories_one_hot = onehot_encoder_categories.transform(df['categories'].apply(eval))
  df_categories_one_hot = pd.DataFrame(categories_one_hot, columns=onehot_encoder_categories.classes_)
  df = pd.concat([df, df_categories_one_hot], axis=1)
  # Engagement features
  df['Likes_Dislikes'] = df['Likes'] - df['Dislikes']
  df_engagement = df[['Visits','Likes_Dislikes']]
  df['Score']= scaler_engagement.transform(df_engagement).sum(axis = 1)/2
  df['engagement'] = df['Score'].apply(categorize_score)
  # Scale features xps, locationLon, locationLat
  df[['xps','locationLon','locationLat']] = scaler_features.transform(df[['xps','locationLon','locationLat']])
  # One hot encoder for tier
  tier_one_hot = onehot_encoder_tier.transform(pd.DataFrame(df['tier']))
  df_tier_one_hot = pd.DataFrame(tier_one_hot, columns=onehot_encoder_tier.get_feature_names_out(['tier']))
  df = pd.concat([df, df_tier_one_hot], axis=1)
  # Remove features
  df_clean = df.drop(['tags','categories','id','name','shortDescription',
                      'Likes','Dislikes','Bookmarks','Visits','Score', 'Likes_Dislikes', 'tier'], axis = 1)
  return df_clean

In [8]:
# apply data processing steps to train, validation and test data
df_train_proc = processdata(df_train)
df_val_proc = processdata(df_val)
df_test_proc = processdata(df_test)

In [ ]:
# train model (check if configuration is ok)
# Prepare Datasets (train and val)
transform_img_norm = transforms.Compose([ # config
    #transforms.Normalize(means, stds)
    #transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize(normalize_images[0], normalize_images[1])
])
train_dataset = img_Dataset(df_train_proc['engagement'], df_train_proc['main_image_path'], transform_img = transform_img_norm)
val_dataset = img_Dataset(df_val_proc['engagement'], df_val_proc['main_image_path'], transform_img = transform_img_norm)
#set_random_seed()
learning_rate = 0.01 # To optimize
dropout_rate = 0.2 # To optimize
batch_size = 64 # To optimize
num_epochs = 8 
criterion = CrossEntropyLoss() 
model = FCNN(dropout_rate) # Optimized parameter
optimizer = Adam(model.parameters(), lr=learning_rate) # Optimized parameter
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) # Optimized parameter
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False) # Optimized parameter
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f'Device: {device}')
train_model(model, criterion, optimizer, num_epochs, train_loader, val_loader, device)

Device: cpu
Epoch 1, Loss: 1.0090353563427925, Acc: 50.0, Val Loss: 0.7486946135759354, Val Acc: 52.58964143426295, LR: 0.01


#### OPTUNA TODO <---

In [9]:
def objective(trial, to_optim = None): # Reference name for the tunning experiment
    """
    Objective function for hyperparameter optimization with Optuna.
    """

    # seed for random numbers
    #set_random_seed() # Useful??
    
    # Prepare Datasets (train and val)
    transform_img_norm = transforms.Compose([ # config
        transforms.ToTensor(),
        #transforms.Normalize(means, stds)
        transforms.Normalize(normalize_images[0], normalize_images[1])
    ])
    train_dataset = img_Dataset(df_train_proc['engagement'], df_train_proc['main_image_path'], transform_img = transform_img_norm)
    val_dataset = img_Dataset(df_val_proc['engagement'], df_val_proc['main_image_path'], transform_img = transform_img_norm)

    # hyperparameters to optimize
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5) # To optimize
    learning_rate = trial.suggest_float("learning_rate", 1e-3, 1e-1, log=True) # To optimize
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256]) # To optimize

    # Neural network configuration
    num_epochs = 2 
    criterion = CrossEntropyLoss() 
    model = CNN(dropout_rate) # Optimized parameter
    optimizer = Adam(model.parameters(), lr=learning_rate) # Optimized parameter
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) # Optimized parameter
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False) # Optimized parameter
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # train model
    train_results = train_model(model, criterion, optimizer, num_epochs,
                                train_loader, val_loader, device, verbose = False)

    # save metrics
    save_metrics_optuna(trial, train_results, outputdir)

    return train_results['val_accs'][-1]

In [ ]:
# Parameters to save results
outputdir = 'kk'
study_id = 'kk'
if not os.path.exists(outputdir):
  os.makedirs(outputdir)
# Number of trials
n_trials = 10
# Optuna initialization
sampler = optuna.samplers.RandomSampler(seed=42)
study = optuna_init(sampler, outputdir, study_id)
# Optuna optimization
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=n_trials, show_progress_bar = True)
# Optuna results
optuna_results(study)

[I 2025-09-18 09:59:53,233] A new study created in RDB with name: kk_optimization


  0%|          | 0/10 [00:00<?, ?it/s]

In [9]:
# Normalize xps, locationLon, locationLat, numTags
scaler_features = StandardScaler().fit(df_train[['xps','locationLon','locationLat']])

# one hot encoder for categories
onehot_encoder_categories = MultiLabelBinarizer().fit(poi_data['categories'].apply(eval))

# One hot encoder for tier
onehot_encoder_tier = OneHotEncoder(sparse_output=False).fit(pd.DataFrame(poi_data['tier']))


NameError: name 'StandardScaler' is not defined

In [ ]:
def processdata(df):
  """
  Data processing steps before being used in the model.
  Same steps are applied to train, validation and test datasets.
  Processing include:
  - Calculate number of tags
  - One hot encoding for categories
  - Calculate engagement feature (low, medium, high)
  - Scale quantitative features
  - One hot encoding for tier
  - Remove features
  """
  df.index = range(df.shape[0])
  # Feature for number of tags
  df['NumTags'] = df['tags'].apply(eval).apply(len)
  # One hot encoder for categories
  categories_one_hot = onehot_encoder_categories.transform(df['categories'].apply(eval))
  df_categories_one_hot = pd.DataFrame(categories_one_hot, columns=onehot_encoder_categories.classes_)
  df = pd.concat([df, df_categories_one_hot], axis=1)
  # Engagement features
  df['Likes_Dislikes'] = df['Likes'] - df['Dislikes']
  df_engagement = df[['Visits','Likes_Dislikes']]
  df['Score']= scaler_engagement.transform(df_engagement).sum(axis = 1)/2
  df['engagement'] = df['Score'].apply(categorize_score)
  # Scale features xps, locationLon, locationLat
  df[['xps','locationLon','locationLat']] = scaler_features.transform(df[['xps','locationLon','locationLat']])
  # One hot encoder for tier
  tier_one_hot = onehot_encoder_tier.transform(pd.DataFrame(df['tier']))
  df_tier_one_hot = pd.DataFrame(tier_one_hot, columns=onehot_encoder_tier.get_feature_names_out(['tier']))
  df = pd.concat([df, df_tier_one_hot], axis=1)
  # Remove features
  df_clean = df.drop(['tags','categories','id','name','shortDescription',
                      'Likes','Dislikes','Bookmarks','Visits','Score', 'Likes_Dislikes', 'tier'], axis = 1)
  return df_clean

In [6]:
# Preprocessing
# Calculate mean and std for each channel of images from train dataset (This will be used for normalization after transform to tensor)
# TODO This could be save in a dictionary
images = read_images(df_train['main_image_path'])
normalize_images = get_normalize_images(images)
# get MinMaxScaler for engagement rows (This will be used for engagement calculation in different dataset
scaler_engagement = get_scaler_engagement(df_train)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1004/1004 [00:12<00:00, 79.58it/s]
